# 13.1 Objects, Names and the Heap

**Prerequisites:** 2.7 Mutability and Copying, 5.1 Python OOPs, 12.1 The GIL  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 **There are no value types** — every object lives on the heap, `int` included
- What a *name* actually is, and why assignment never copies
- `id()`, `is` and `==` — which question each one answers
- Interning: the small-int cache, string literals, and `sys.intern`
- 🔴 **Immortal objects** (3.12+) and why old refcount tutorials now lie
- Why `locals()` is a snapshot but `globals()` is the real thing
- ⚠️ *heap* the memory region vs `heapq` the data structure — same word, unrelated

---

## The model you probably brought with you

If you have written C, C++, Java or C#, you carry a rule that sounds like this:

> *Primitives live on the stack. Objects live on the heap. Assignment copies the primitive
> and copies the reference.*

That rule is real in those languages and load-bearing. **In CPython it is simply false**, and
carrying it over produces confident, wrong predictions — "small ints are cheap because
they're on the stack", "passing a config dict copies it", "reassigning frees the old value".

The actual rule is shorter:

> **Every Python object is allocated on the heap. A name is a pointer to one. Assignment
> binds a name; it never copies an object.**

That is the whole model. Everything in this folder follows from it.

> ⚠️ **Two meanings of "heap".** Here, *heap* means the memory region the allocator hands
> objects out of. In **14.8** *heap* means a binary-heap priority queue (`heapq`). They share a
> name and nothing else. This notebook only ever means the memory region.

## Names point; they do not hold

The clearest demonstration is a config object shared by two names.

In [ ]:
import sys

# Two names, one object. Nothing was copied.
primary = {"host": "db-1", "port": 5432, "role": "primary"}
replica = primary

print("primary is replica :", primary is replica)
print("same id()          :", id(primary) == id(replica))

replica["port"] = 5433                      # mutate through one name
print("read back through the other name:", primary)

print("\nEvery one of these is a heap object with an address:")
sentinel = object()
for value, label in [(5432, "int   (a port)"),
                     ("db-1", "str   (a host)"),
                     (["db-1", "db-2"], "list  (a replica set)"),
                     (("db-1", 5432), "tuple (an endpoint)"),
                     (sentinel, "object (a sentinel)")]:
    print(f"  {label:24} id -> {id(value):>16}")

Both names refer to the *same dictionary*, so the write through `replica`
is visible through `primary`. No copy exists to be out of date.

The second block is the part that surprises people: **the port number has an address too.**
`5432` is a heap-allocated `int` object, not a machine word sitting in a slot. So is the
string, the list, the tuple, and the bare sentinel.

### `id()`, `is` and `==`

| Ask | Question it answers | Use it for |
|---|---|---|
| `id(x)` | *which object is this?* | debugging identity; the number itself is meaningless |
| `x is y` | *are these the same object?* | 🔴 `None`, `True`, `False`, sentinels — nothing else |
| `x == y` | *do these have the same value?* | 🔴 everything you actually care about |

🔴 **`is` on numbers or strings is a bug that passes its tests.** The next cell shows exactly
how it fools you.

## Interning — why `is` sometimes lies

CPython reuses objects for common values. Two separate optimisations do this, and neither is
part of the language spec:

- **The small-int cache** — every `int` from **-5 to 256** is created once at startup and reused.
- **Compile-time constant folding** — identical literals *inside one code object* become one
  shared constant, whatever their value.

The second one is why the classic "`257 is not 257`" demo fails inside a single cell. To see
real behaviour you have to build the values at **runtime**, the way parsing a config file does.

In [ ]:
# --- literals in the same code object: the compiler shares them ---
a = 257
b = 257
print("a = 257; b = 257    -> a is b :", a is b, "  <- ONE folded constant, not two ints")

# --- built at runtime, as config parsing actually does ---
raw_config = {"port": "257", "pool_size": "256", "retry_delay": "-5", "backoff": "-6"}

port_1, port_2 = int(raw_config["port"]), int(raw_config["port"])
pool_1, pool_2 = int(raw_config["pool_size"]), int(raw_config["pool_size"])
delay_1, delay_2 = int(raw_config["retry_delay"]), int(raw_config["retry_delay"])
back_1, back_2 = int(raw_config["backoff"]), int(raw_config["backoff"])

print("\nparsed twice from config, so the compiler cannot help:")
print(f'  int("257") -> is : {port_1 is port_2}    (outside the cache)')
print(f'  int("256") -> is : {pool_1 is pool_2}    (cached)')
print(f'  int("-5")  -> is : {delay_1 is delay_2}    (cached)')
print(f'  int("-6")  -> is : {back_1 is back_2}    (outside the cache)')

# --- find the boundary by measurement, not by trusting the docs ---
cached = [n for n in range(-1000, 1000) if int(str(n)) is int(str(n))]
print(f"\n  measured cache range: {min(cached)} .. {max(cached)}")

# --- the same trap with strings ---
literal = "deploy"
assembled = "".join(["dep", "loy"])          # built at runtime, e.g. from a log line
print("\n  assembled is literal        :", assembled is literal)
print("  sys.intern(assembled) is ...:", sys.intern(assembled) is literal)
print("  assembled == literal        :", assembled == literal, " <- the question you meant")

Read the first line carefully: `a is b` is **`True`** for 257, which looks
like it contradicts everything you have heard. It does not — the compiler saw two identical
literals in one code object and stored **one** constant. Both names point at it.

Parse the same values out of a config dict and the truth appears: `256` and `-5` come back
identical because they are cached; `257` and `-6` come back as distinct objects. The measured
range is exactly **-5 to 256**.

The string case is the one that bites in production. A hostname you *typed* and a hostname you
**built from a log line** compare equal but are not the same object — so an `is` check that
passed every test against literals fails the moment the value arrives from a file, a socket or
a database.

> **`sys.intern()`** forces a string into the interning table. It is worth knowing for one real
> case: millions of repeated keys (log fields, column names) where dedup saves genuine memory.
> It is never a substitute for `==`.

## Immortal objects — and why old tutorials now lie

CPython tracks how many references point at each object (**13.3** covers this properly). Every
tutorial written before 2023 demonstrates it like this: *take a small integer, watch its
refcount rise as you assign it.*

That demonstration **no longer works**. Since **Python 3.12** (PEP 683) the objects that are
never worth freeing — `None`, `True`, `False`, the small-int cache, and some interned strings —
are **immortal**: their reference count is pinned to a huge sentinel value and never changes.
This is what makes free-threaded builds (**12.1**) viable, since immortal objects need no
locking to refcount.

So a refcount at or above `2**30` does not mean "a billion references". It means *this object
will never be freed*.

In [ ]:
IMMORTAL_FLOOR = 2 ** 30

print("is this object immortal?")
for obj, label in [(None, "None"),
                   (True, "True"),
                   (200, "small int 200 (cached)"),
                   (5432, "int 5432 (not cached)"),
                   (["db-1"], "a fresh list")]:
    count = sys.getrefcount(obj)
    verdict = "IMMORTAL - never freed" if count >= IMMORTAL_FLOOR else f"mortal, refcount {count}"
    print(f"  {label:26} {verdict}")

# A mortal object, counted precisely. getrefcount() sees its own argument,
# so every number below includes +1 for the call itself.
print("\ntracking one task record through a work queue:")
task = {"id": "task-91", "state": "queued"}
print("  just created                 :", sys.getrefcount(task))

queue = [task]
print("  after queue.append           :", sys.getrefcount(task))

index = {"task-91": task}
print("  after indexing it by id      :", sys.getrefcount(task))

queue.clear()
print("  after queue.clear()          :", sys.getrefcount(task))

del index["task-91"]
print("  after dropping from the index:", sys.getrefcount(task))
print("  ^ back to the start: one name (`task`) plus getrefcount's own argument")

`None`, `True` and the cached `200` report as immortal. `5432` — outside
the small-int cache — is an ordinary mortal object, as is the fresh list.

The task record tells the real story: **2** when only `task` names it, **3** once the queue
holds it, **4** once the index holds it too, and back down as each container lets go. Every
count includes `+1` for the argument `getrefcount` is holding while it runs, which is the
single most common reason people report the "wrong" number.

🔴 That last point matters for real code: as long as *any* container still references the task,
it stays alive. A queue you forgot to clear, or a cache with no eviction, is exactly how a
service leaks memory while looking correct — **13.3** shows how to find and fix that.

## What a name actually is

A name is an entry in a namespace. At module level that namespace is a real dictionary you can
inspect and even write to. Inside a function it is **not** — locals are compiled into numbered
slots in the frame for speed, and `locals()` hands you a *snapshot*, not the real thing.

In [ ]:
DEPLOY_ENV = "staging"

print("module scope is a real dict:")
print("  'DEPLOY_ENV' in globals() :", "DEPLOY_ENV" in globals())
globals()["FEATURE_FLAG"] = "checkout-v2"          # a name with no assignment statement
print("  FEATURE_FLAG              :", FEATURE_FLAG)


def handle_request(request_id):
    """Function scope is compiled to slots, not a dict."""
    retries = 3
    snapshot = locals()
    snapshot["retries"] = 999                       # write into the snapshot...
    return retries, snapshot["retries"]             # ...the real local is untouched


actual, in_snapshot = handle_request("req-7")
print("\nfunction scope is not:")
print("  real local after editing locals():", actual)
print("  value inside the snapshot        :", in_snapshot)
print("  -> locals() is a copy; the real slots live in the frame (13.2)")

print("\ndeleting a name does not delete the object:")
db_config = {"region": "eu-west-1"}
retained_by = db_config                             # a second reference
del db_config
print("  'db_config' still a name? :", "db_config" in globals())
print("  object still alive        :", retained_by)

Writing into `globals()` creates a usable module-level name with no
assignment statement anywhere — which is exactly how plugin loaders and `from x import *`
work.

The function case is the interesting one. Editing the dict returned by `locals()` changes
**nothing**: the real local is still `3`, while the snapshot says `999`. Locals live in the
frame's slots, and `locals()` only ever copies them out.

> **Version note.** PEP 667 (**Python 3.13**) made this explicit. `locals()` in a function has
> always returned an independent snapshot in CPython; from 3.13 that is the *documented,
> guaranteed* behaviour, and debuggers use `frame.f_locals` for a live view instead (**13.2**).

And `del` removes the *name*, not the object. The dict survives because another name still
points at it. **This is the whole of Python's memory model in one line** — objects live until
nothing refers to them, which is precisely what **13.3** is about.

---

## Common Mistakes & Pitfalls

1. 🔴 **Using `is` to compare values.** It works on literals and fails the moment the value is parsed from config, a socket or a database. Use `==`; reserve `is` for `None`, `True`, `False` and sentinels.
2. 🔴 **Believing `a = 257; b = 257; a is b` proves ints are cached.** That is *constant folding*, not the cache. Build the values at runtime to see real behaviour.
3. **Assuming assignment copies.** `replica = primary` gives you a second name for one object. For a real copy see **2.7**.
4. **Reading a refcount as a reference count.** It includes `getrefcount`'s own argument, and for immortal objects it is a fixed sentinel, not a count.
5. 🔴 **Quoting a pre-3.12 refcount tutorial.** Small ints, `None` and `True` are immortal now; their counts never move.
6. **Editing `locals()` and expecting the function to notice.** It is a snapshot (PEP 667).
7. **Thinking `del x` frees memory.** It unbinds a name. The object survives while anything else references it.
8. **Confusing the memory heap with `heapq`.** Unrelated (**14.8**).
9. **Reaching for `sys.intern` as an optimisation.** It pays off only for very large numbers of repeated strings, and never replaces `==`.

## Best Practices

- Compare with `==`. Keep `is` for `None`, `True`, `False` and sentinel objects.
- Treat `id()` values as opaque — useful for *are these the same?*, never for arithmetic.
- When you need an independent object, copy it deliberately (**2.7**), at the boundary.
- Use a unique `object()` as a sentinel when `None` is a legitimate value (**4.1**).
- Remember any container holding a reference keeps the object alive — clear queues and bound caches (**13.3**).
- Read `frame.f_locals` in a debugger, not `locals()`, when you need a live view (**13.2**).
- Assume nothing about interning. It is a CPython implementation detail, not a language guarantee.

## Practice Exercises

Try these before moving on.

1. Predict, then check: does `int(a) is int(b)` hold for `"10"`, `"256"`, `"257"`? Explain each result.
2. Find the small-int cache boundary by measurement, as the notebook does. Does it move between 3.12, 3.13 and 3.14 on your machine?
3. 🔴 Write a function that compares two hostnames with `is`. Make it pass with literals, then break it by reading one hostname from a file. This is a real bug pattern.
4. Take a task record, put it in a list, a dict and a set of one-element tuples. Track `getrefcount` after each. Explain every number, including the `+1`.
5. Show that `del` on one of two names leaves the object alive, then drop the second name and show it is collected (**13.3** gives you the tool).
6. 🔴 Build a dict of 200,000 log records whose `level` field is a fresh string each time. Measure `tracemalloc` before and after `sys.intern`-ing that field (**17.5**).
7. Explain to someone with a Java background why `Integer.valueOf` caching and Python's small-int cache are the same idea, and why neither should be relied on.
8. **Interview question:** *“Is everything in Python a reference?”* Answer it precisely, using `id`, assignment and mutation.

---

## Version notes

| Version | Change |
|---|---|
| **3.12** | 🔴 **PEP 683 immortal objects** — `None`, `True`, `False`, small ints and some interned strings get a pinned refcount that never changes |
| **3.12** | Comprehensions are inlined, so they no longer create a separate frame (**13.2**) |
| **3.13** | 🔴 **PEP 667** — `locals()` in a function is *documented* as an independent snapshot; `frame.f_locals` gives the live view |
| **3.13** | The free-threaded build (no GIL) becomes available; immortality is part of what makes it possible (**12.1**) |
| **3.14** | Further free-threading work; the object model in this notebook is unchanged |

> 🔴 **All of this is CPython, not Python.** The language spec guarantees none of it.
> PyPy has no reference counting at all and its `id()` is a fabricated number; MicroPython and
> GraalPy differ again. Write code that works on any of them: compare with `==`, do not depend
> on interning, and never make `is` load-bearing for a value.

## 13 How Python Works Under the Hood — the folder

| Notebook | Covers |
|---|---|
| **13.1** | this notebook — objects on the heap, names, identity, interning, immortality |
| **13.2** | the call stack, frames, recursion limits, why locals are not "on the stack" |
| **13.3** | reference counting, cycles, the garbage collector, `weakref` |
| **13.4** | where the memory actually goes — `getsizeof`, container overhead, `__slots__` measured |
| **13.5** | attribute lookup and class creation — descriptors, `__init_subclass__`, metaclasses |

**The one-sentence version:** *every object is on the heap, a name is a pointer to one, and
nothing is ever copied unless you ask.*

## Related

- **2.7** Mutability, Copying, Nesting and Unpacking — the practical consequences of this model
- **5.1** Python OOPs — where `__slots__` is introduced; **13.4** measures it
- **12.1** The GIL — why refcounting and free-threading interact
- **14.8** Heaps and Priority Queues — the *other* heap